In [151]:
import pandas as pd
import numpy as np
import numpy.linalg as la
import pickle
from pykalman import KalmanFilter

In [142]:
# def add_gaussian_noise_to_signal(signal, mean=0, std=1):
#     noisy_signal = signal.copy()
#     noisy_signal['depth'] = noisy_signal['depth'] + np.random.normal(mean, std, len(noisy_signal['depth']))
#     return noisy_signal
# 
# df['noisy_signal'] = df['decrease'].apply(lambda x: add_gaussian_noise_to_signal(x, mean=0, std=0.1))  # Adjust `std` as needed

In [143]:
df = pd.read_pickle("flood_df")
df.head()

,deployment_id,label,inflection_t,signal,signal_padded,signal_sim,signal_sim_padded
uuid,,,,,,,
1690089,daily_happy_satyr,flood,6961,"{'time': [0, 567, 1009, 1639, 1765, 2080, 2395...","{'time': [0, 60, 120, 180, 240, 300, 867, 1309...","{'time': [0, 567, 1009, 1639, 1765, 2080, 2395...","{'time': [0, 60, 120, 180, 240, 300, 867, 1309..."
2578925,daily_happy_satyr,flood,7062,"{'time': [0, 1889, 1952, 2960, 4914, 4921, 674...","{'time': [0, 60, 120, 180, 240, 300, 2189, 225...","{'time': [0, 1889, 1952, 2960, 4914, 4921, 674...","{'time': [0, 60, 120, 180, 240, 300, 2189, 225..."
5301386,daily_happy_satyr,flood,5303,"{'time': [0, 1323, 1898, 1906, 2409, 2535, 278...","{'time': [0, 60, 120, 180, 240, 300, 1623, 219...","{'time': [0, 1323, 1898, 1906, 2409, 2535, 278...","{'time': [0, 60, 120, 180, 240, 300, 1623, 219..."
2808962,daily_happy_satyr,flood,4789,"{'time': [126, 441, 503, 566, 882, 1008, 1071,...","{'time': [0, 60, 120, 180, 240, 300, 615, 677,...","{'time': [126, 441, 503, 566, 882, 1008, 1071,...","{'time': [0, 60, 120, 180, 240, 300, 615, 677,..."
9041605,daily_happy_satyr,flood,6188,"{'time': [0, 379, 1449, 1575, 1890, 2394, 2709...","{'time': [0, 60, 120, 180, 240, 300, 679, 1749...","{'time': [0, 379, 1449, 1575, 1890, 2394, 2709...","{'time': [0, 60, 120, 180, 240, 300, 679, 1749..."


In [144]:
def preprocess_signal(signal):
    time = signal['time']
    depth = signal['depth']
    return pd.DataFrame({'time': time, 'depth': depth}).sort_values(by='time')

df['processed_signal'] = df['signal_padded'].apply(preprocess_signal)
print(df['processed_signal'].iloc[0])

     time  depth
0       0    0.0
1      60    0.0
2     120    0.0
3     180    0.0
4     240    0.0
5     300    4.0
6     867   18.0
7    1309   25.0
8    1939   35.0
9    2065   37.0
10   2380   50.0
11   2695   64.0
12   2947   79.0
13   3010   82.0
14   3136   90.0
15   3387  110.0
16   3766  139.0
17   4018  160.0
18   4080  164.0
19   4332  181.0
20   5215  237.0
21   5781  266.0
22   6033  274.0
23   6229  283.0
24   6489  292.0
25   6560  296.0
26   6568  296.0
27   6694  298.0
28   6757  298.0
29   6820  300.0
30   7198  307.0
31   7261  308.0
32   7386  310.0
33   7575  307.0
34   7638  302.0
35   8142  297.0
36   8269  292.0
37   9088  268.0
38   9151  266.0
39   9403  260.0
40  10033  240.0
41  10536  220.0
42  10915  204.0
43  11230  189.0
44  11734  162.0
45  12238  131.0
46  12868   85.0
47  12993   74.0
48  13182   59.0
49  13497   34.0
50  13560   24.0
51  13812    2.0
52  13872    0.0
53  13932    0.0
54  13992    0.0
55  14052    0.0
56  14112    0.0


In [145]:
def vectorized_kalman_filter_3d(signal):
    observations = signal['depth'].values
    times = signal['time'].values
    time_diffs = np.diff(times, prepend=times[0])  # Compute time gaps with 0 for the first element

    # Initialize state and covariance matrices
    n_observations = len(observations)
    state_dim = 3  # [depth, velocity, acceleration]
    filtered_means = np.zeros((n_observations, state_dim))
    smoothed_means = np.zeros((n_observations, state_dim))
    covariances = np.zeros((n_observations, state_dim, state_dim))

    # Initial state and covariance
    state_mean = np.array([observations[0], 0, 0])  # Initial [depth, velocity, acceleration]
    state_covariance = np.eye(state_dim) * 1  # Initial uncertainty
    observation_matrix = np.array([[1, 0, 0]])  # Observation model
    observation_covariance = np.array([[1]])  # Observation noise
    process_noise_base = np.array([[6, 0.05, 0.0], [6, 0.1, 0.0], [0.0, 0.0, 0.0]]) / 10

    epsilon = 1e-6  # Small value for regularization

    # Filtering
    for t in range(n_observations):
        if t > 0:
            delta_t = time_diffs[t - 1]
            transition_matrix = np.array([
                [1, delta_t, 0.5 * delta_t**2],
                [0, 1, delta_t],
                [0, 0, 1]
            ])
            process_noise = process_noise_base * delta_t

            # Predict step
            predicted_state_mean = np.dot(transition_matrix, state_mean)
            predicted_state_cov = (
                np.dot(transition_matrix, np.dot(state_covariance, transition_matrix.T)) + process_noise
            )

            # Update step
            innovation = observations[t] - np.dot(observation_matrix, predicted_state_mean)
            innovation_cov = (
                np.dot(observation_matrix, np.dot(predicted_state_cov, observation_matrix.T))
                + observation_covariance
            )
            kalman_gain = np.dot(
                predicted_state_cov,
                np.dot(observation_matrix.T, np.linalg.inv(innovation_cov + epsilon * np.eye(innovation_cov.shape[0])))
            )
            state_mean = predicted_state_mean + np.dot(kalman_gain, innovation)
            state_covariance = predicted_state_cov - np.dot(
                kalman_gain, np.dot(observation_matrix, predicted_state_cov)
            )

        # Store filtered results
        filtered_means[t] = state_mean
        covariances[t] = state_covariance

    # Smoothing
    smoothed_means[-1] = filtered_means[-1]
    smoothed_covariance = covariances[-1]
    for t in range(n_observations - 2, -1, -1):
        delta_t = time_diffs[t]
        transition_matrix = np.array([
            [1, delta_t, 0.5 * delta_t**2],
            [0, 1, delta_t],
            [0, 0, 1]
        ])

        # RTS smoother gain
        predicted_covariance = (
            np.dot(transition_matrix, np.dot(covariances[t], transition_matrix.T)) + process_noise_base * delta_t
        )
        predicted_covariance += epsilon * np.eye(predicted_covariance.shape[0])  # Regularization
        smoother_gain = np.dot(
            covariances[t],
            np.dot(transition_matrix.T, np.linalg.pinv(predicted_covariance))  # Use pseudo-inverse
        )

        # Update smoothed state
        smoothed_means[t] = (
            filtered_means[t]
            + np.dot(smoother_gain, (smoothed_means[t + 1] - np.dot(transition_matrix, filtered_means[t])))
        )

    # Add filtered and smoothed results to the signal DataFrame
    signal['kalman_depth'] = filtered_means[:, 0]
    signal['smoothed_depth'] = smoothed_means[:, 0]
    signal['velocity'] = filtered_means[:, 1]
    signal['acceleration'] = filtered_means[:, 2]
    return signal


In [146]:
df['filtered_signal'] = df['processed_signal'].apply(vectorized_kalman_filter_3d)

In [147]:
print(df['filtered_signal'].iloc[4])

     time  depth  kalman_depth  smoothed_depth    velocity  acceleration
0       0    0.0      0.000000        0.000125    0.000000      0.000000
1      60    0.0      0.000000        0.000125    0.000000      0.000000
2     120    0.0      0.000000       -0.006302    0.000000      0.000000
3     180    0.0      0.000000        0.016142    0.000000      0.000000
4     240    0.0      0.000000       -0.001312    0.000000      0.000000
5     300   10.0     10.004474       10.001109    0.010131      0.000051
6     679   17.0     17.002991       17.001784    0.012355      0.000048
7    1749   44.0     44.000060       43.999365    0.066822      0.000048
8    1875   49.0     49.000068       49.000736   -0.013043      0.000048
9    2190   61.0     60.999923       60.999085    0.104017      0.000048
10   2694   81.0     81.000237       81.000247    0.026515      0.000048
11   3009   98.0     98.000004       97.999704    0.044716      0.000048
12   3072  101.0    101.000031      101.122826    0

In [148]:
def improved_em_for_kalman(signal_set, max_iterations=100, tolerance=1e-6):
    """
    Improved Expectation-Maximization (EM) algorithm for estimating global Kalman filter parameters.
    
    Parameters:
    -----------
    signal_set : list of DataFrames
        List of signals, each containing 'time' and 'depth' columns
    max_iterations : int, optional
        Maximum number of EM iterations
    tolerance : float, optional
        Convergence threshold for parameter changes
    
    Returns:
    --------
    dict
        Dictionary containing estimated global Kalman filter parameters
    """
    # Dimension constants
    state_dim = 3  # [depth, velocity, acceleration]
    obs_dim = 1    # Observing only depth
    
    # Initialization with robust initial guesses
    def robust_initial_guess(signals):
        # Compute initial guesses based on signal characteristics
        depths = [signal['depth'].values for signal in signals]
        times = [signal['time'].values for signal in signals]
        
        # Compute mean and variance of depths
        all_depths = np.concatenate(depths)
        mean_depth = np.mean(all_depths)
        std_depth = np.std(all_depths)
        
        # Initial transition matrix with time scaling
        mean_time_diff = np.mean([np.mean(np.diff(time)) for time in times])
        
        transition_matrix = np.array([
            [1, mean_time_diff, 0.5 * mean_time_diff**2],
            [0, 1, mean_time_diff],
            [0, 0, 1]
        ])
        
        # Initial noise matrices with scaled variance
        process_noise_cov = np.eye(state_dim) * (std_depth / 10)**2
        observation_noise_cov = np.array([[std_depth / 5]])
        observation_matrix = np.array([[1, 0, 0]])
        
        return transition_matrix, process_noise_cov, observation_matrix, observation_noise_cov
    
    # Initial parameter estimation
    transition_matrix, process_noise_cov, observation_matrix, observation_noise_cov = robust_initial_guess(signal_set)
    
    # Regularization constants
    epsilon = 1e-8
    
    # Iteration tracking
    for iteration in range(max_iterations):
        # Storage for sufficient statistics
        sum_x_prev_x = np.zeros((state_dim, state_dim))
        sum_x_curr_x = np.zeros((state_dim, state_dim))
        sum_obs_x = np.zeros((obs_dim, state_dim))
        sum_obs_obs = 0
        total_obs_count = 0
        
        # E-Step: Process each signal
        for signal in signal_set:
            # Apply Kalman filtering and smoothing
            observations = signal['depth'].values
            times = signal['time'].values
            
            # Time difference computation
            delta_times = np.diff(times, prepend=times[0])
            
            # Kalman filtering with flexible time-dependent transition
            filtered_means = np.zeros((len(observations), state_dim))
            filtered_covs = np.zeros((len(observations), state_dim, state_dim))
            
            # Initial state
            filtered_means[0] = np.array([observations[0], 0, 0])
            filtered_covs[0] = np.eye(state_dim)
            
            # Forward filtering
            for t in range(1, len(observations)):
                # Adaptive transition matrix based on time difference
                delta_t = delta_times[t]
                local_transition = np.array([
                    [1, delta_t, 0.5 * delta_t**2],
                    [0, 1, delta_t],
                    [0, 0, 1]
                ])
                
                # Prediction
                pred_mean = local_transition @ filtered_means[t-1]
                pred_cov = (local_transition @ filtered_covs[t-1] @ local_transition.T) + process_noise_cov
                
                # Update
                innovation = observations[t] - observation_matrix @ pred_mean
                innovation_cov = observation_matrix @ pred_cov @ observation_matrix.T + observation_noise_cov
                
                # Kalman gain
                kalman_gain = pred_cov @ observation_matrix.T @ la.inv(innovation_cov + epsilon * np.eye(obs_dim))
                
                filtered_means[t] = pred_mean + kalman_gain @ innovation
                filtered_covs[t] = (np.eye(state_dim) - kalman_gain @ observation_matrix) @ pred_cov
            
            # Backwards smoothing (RTS smoother)
            smoothed_means = filtered_means.copy()
            for t in range(len(observations)-2, -1, -1):
                delta_t = delta_times[t+1]
                local_transition = np.array([
                    [1, delta_t, 0.5 * delta_t**2],
                    [0, 1, delta_t],
                    [0, 0, 1]
                ])
                
                # Smoother gain
                smoother_gain = filtered_covs[t] @ local_transition.T @ la.inv(local_transition @ filtered_covs[t] @ local_transition.T + process_noise_cov)
                
                smoothed_means[t] += smoother_gain @ (smoothed_means[t+1] - local_transition @ smoothed_means[t])
            
            # Accumulate sufficient statistics
            for t in range(1, len(observations)):
                x_prev = smoothed_means[t-1]
                x_curr = smoothed_means[t]
                obs = observations[t]
                
                sum_x_prev_x += np.outer(x_prev, x_curr)
                sum_x_curr_x += np.outer(x_curr, x_curr)
                sum_obs_x += obs * x_curr
                sum_obs_obs += obs**2
                total_obs_count += 1
        
        # M-Step: Update global parameters
        # Transition matrix
        new_transition_matrix = sum_x_prev_x @ la.pinv(sum_x_curr_x + epsilon * np.eye(state_dim))
        
        # Process noise
        new_process_noise_cov = (sum_x_curr_x - new_transition_matrix @ sum_x_prev_x.T) / total_obs_count
        new_process_noise_cov = np.maximum(new_process_noise_cov, epsilon * np.eye(state_dim))
        
        # Observation matrix
        new_observation_matrix = sum_obs_x @ la.pinv(sum_x_curr_x + epsilon * np.eye(state_dim))
        
        # Observation noise
        new_observation_noise_cov = (sum_obs_obs - np.trace(new_observation_matrix @ sum_obs_x.T)) / total_obs_count
        new_observation_noise_cov = max(new_observation_noise_cov, epsilon)
        
        # Convergence check
        param_changes = [
            la.norm(transition_matrix - new_transition_matrix),
            la.norm(process_noise_cov - new_process_noise_cov),
            la.norm(observation_matrix - new_observation_matrix),
            abs(observation_noise_cov - new_observation_noise_cov)
        ]
        
        # Update parameters
        transition_matrix = new_transition_matrix
        process_noise_cov = new_process_noise_cov
        observation_matrix = new_observation_matrix
        observation_noise_cov = new_observation_noise_cov
        
        # Check for convergence
        if all(change < tolerance for change in param_changes):
            break
    
    # Return estimated global parameters
    return {
        'transition_matrix': transition_matrix,
        'process_noise_cov': process_noise_cov,
        'observation_matrix': observation_matrix,
        'observation_noise_cov': observation_noise_cov
    }


In [149]:
filtered_signals = df['filtered_signal'].tolist()
global_params = improved_em_for_kalman(filtered_signals)

In [150]:
print(global_params)

{'transition_matrix': array([[ 9.99302690e-01, -8.89254107e+01,  1.20475193e+04],
       [ 1.47139571e-06,  9.84531720e-01, -7.37865554e+01],
       [-7.43101391e-10,  1.46170704e-04,  9.94547447e-01]]), 'process_noise_cov': array([[5.11599747e+01, 6.30470362e-03, 0.00000000e+00],
       [6.30470362e-03, 8.93516597e-06, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.00000000e-08]]), 'observation_matrix': array([[ 1.00000001e+00, -4.14417556e-04, -1.67754300e-02]]), 'observation_noise_cov': 1e-08}


In [152]:
with open('global_params.pkl', 'wb') as file:
    pickle.dump(global_params, file)